In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import keras
from keras import layers
import pprint
import logging
from logging import info,warning,error,debug
from keras.src import ops
import PPO_Model2 as pmodel
format="%(asctime)s %(levelname)s %(message)s"
logging.basicConfig(format=format,level=logging.DEBUG,force=1)




##(по желанию) считал load-balancing по top_k_value



In [5]:
class Transformer_block(layers.Layer):
    def __init__(self,num_head,key_dim,d_ff,num,batch_dim,value_dim=None,dropout=0.1,l=None,
                  num_experts=1,top_k=1,*, activity_regularizer=None, trainable=True, dtype=None, autocast=True,
                    name=None, **kwargs):
        super().__init__(activity_regularizer=activity_regularizer, trainable=trainable, dtype=dtype, autocast=autocast, name=name, **kwargs)
        self.mha=pmodel.castom_MHA(num_head,key_dim,value_dim,dropout=dropout,activity_regularizer=keras.activations.gelu)
        self.moe=pmodel.Moe(d_ff,num,num_experts,top_k,l,batch_dim)
        
    def build(self, input_shape):
        self.norm0=pmodel.RMSNorm(input_shape[-1])
        self.norm1=pmodel.RMSNorm(input_shape[-1])
        return super().build(input_shape)
    
    def call(self,x,training=True):
        attention=self.mha(x,x,training=training)
        attention+=x
        attention=self.norm0(attention)
        y=self.moe(attention,training=training)
        y+=attention
        y=self.norm1(y)
        return y




In [2]:
np.set_printoptions(
    linewidth=np.inf,
    suppress=True,
    threshold=np.inf,
    precision=3,
)
inp=layers.Input([150,64])
outp=pmodel.Transformer_block(num_head=5,key_dim=128,d_ff=128,num=1,
                       batch_dim=32,value_dim=None,dropout=0.1,l=1,num_experts=8,top_k=2)(inp,)
x=tf.ones((32,150,64))
a=keras.Model(inp,outp)
a(x)


2026-02-04 20:02:36,398 WARNING From d:\Python\Lib\site-packages\keras\src\backend\tensorflow\core.py:232: The name tf.placeholder is deprecated. Please use tf.compat.v1.placeholder instead.



<tf.Tensor: shape=(32, 150, 64), dtype=float32, numpy=
array([[[ 0.943,  0.678,  0.877,  1.131,  0.45 ,  1.093,  0.622,  0.99 ,  0.413,  1.44 ,  0.647,  0.818,  1.022,  1.002,  0.786,  1.371,  0.825,  1.099,  0.598,  1.06 ,  0.58 ,  0.995,  0.554,  1.172,  0.872,  0.712,  1.289,  0.999,  1.001,  1.281,  1.111,  1.311,  1.223,  1.216,  1.183,  1.306,  1.313,  0.923,  0.268,  1.463,  0.451,  0.811,  0.556,  0.797,  0.976,  1.37 ,  1.155,  0.785,  1.628,  1.086,  0.658,  0.963,  1.29 ,  0.965,  0.114,  0.357,  1.611,  0.805,  0.901,  1.02 ,  0.437,  0.793,  0.937,  1.374],
        [ 1.249,  0.282,  1.116,  1.391, -0.145,  0.855,  0.588,  1.007,  0.628,  1.418,  0.531,  0.647,  0.703,  0.676,  0.846,  1.07 ,  0.534,  1.028,  0.7  ,  1.054,  0.571,  1.036,  0.765,  0.861,  0.914,  0.915,  1.383,  0.706,  0.721,  1.534,  0.879,  1.238,  1.413,  1.025,  1.258,  1.069,  1.088,  1.121,  0.54 ,  1.73 ,  0.387,  0.67 ,  0.525,  0.919,  0.699,  1.318,  1.299,  0.918,  1.647,  1.126,  1.023,  0.768

In [12]:
a=tf.ones((10,5,6,3))
print(tf.reshape(a,[10,-1,3]))

tf.Tensor(
[[[1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]]

 [[1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]]

 [[1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1

In [ ]:
---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
Cell In[2], line 8
      1 np.set_printoptions(
      2     linewidth=np.inf,
      3     suppress=True,
      4     threshold=np.inf,
      5     precision=3,
      6 )
      7 inp=layers.Input([150,64])
----> 8 outp=pmodel.Transformer_block(num_head=5,key_dim=128,d_ff=128,num=1,
      9                        batch_dim=32,value_dim=None,dropout=0.1,l=1,num_experts=8,top_k=2)(inp,)
     10 x=tf.ones((32,150,64))
     11 a=keras.Model(inp,outp)

File d:\Python\Lib\site-packages\tensorflow\python\util\traceback_utils.py:153, in filter_traceback.<locals>.error_handler(*args, **kwargs)
    151 except Exception as e:
    152   filtered_tb = _process_traceback_frames(e.__traceback__)
--> 153   raise e.with_traceback(filtered_tb) from None
    154 finally:
    155   del filtered_tb

File d:\Python\Lib\site-packages\tensorflow\python\framework\func_graph.py:1048, in func_graph_from_py_func.<locals>.convert(x)
   1046     x = ops.convert_to_tensor_or_composite(x)
   1047   except (ValueError, TypeError):
-> 1048     raise TypeError(
   1049         "To be compatible with tf.function, Python functions "
   1050         "must return zero or more Tensors or ExtensionTypes or None "
   1051         f"values; in compilation of {str(python_func)}, found return "
   1052         f"value of type {type(x).__name__}, which is not a Tensor or "
   1053         "ExtensionType.")
   1054 if add_control_dependencies:
   1055   x = deps_ctx.mark_as_return(x)

TypeError: To be compatible with tf.function, Python functions must return zero or more Tensors or ExtensionTypes or None values; in compilation of <function Transformer_block at 0x000001CCD7603560>, found return value of type Transformer_block, which is not a Tensor or ExtensionType.

SyntaxError: invalid syntax (280846967.py, line 1)